In [ ]:
import matplotlib.pyplot as plt 
import numpy as np 
from nbodykit.lab import ArrayCatalog, FFTPower, BigFileMesh
import pandas as pd

In [ ]:
# Simulation parameters

BoxSize = 1000
Ngrid = 512
kf = 2*np.pi/BoxSize

Verbose=False

# Numbers of bin to compute the power spectra
nbins = 20

In [ ]:
'''
Load of our data. The data variable stores an array with the columns "x y z delta", 
where delta is the overdensity computed as shown in the respective file
'''

data = np.genfromtxt("/path/to/your/file", 
                     delimiter=" ", skip_header=0, names=True)
x, y, z, delta = data['x'], data['y'], data['z'], data['delta'] 

'''
Here, we divide our data in delta bins, defined to have the same number of objects.
'''

df = pd.DataFrame({'x': x, 'y': y, 'z': z, 'delta': delta})
df['bin'], bin_edges = pd.qcut(df['delta'], q=nbins, labels=False, retbins=True, duplicates='drop')
route_bins = "/file/to/store/each/bin.txt"
bin_limits = np.column_stack((bin_edges[:-1], bin_edges[1:]))  
np.savetxt(route_bins, bin_limits, delimiter=" ", header="lim_inf lim_sup")

In [ ]:
'''
Power spectra computation for the halos, P_hh
'''

P_k = []
k_halos = []

N = np.zeros(nbins)     # Variable to store the number of objects for each bin

def phh_compute(idx, x, y, z, Ngrid, BoxSize):
    pos_bin = np.array([x[idx], y[idx], z[idx]]).T

    catalog = ArrayCatalog({'Position': pos_bin})
    mesh = catalog.to_mesh(resampler='cic', compensated=True, Nmesh=Ngrid, BoxSize=BoxSize)

    r = FFTPower(mesh, mode='1d', dk=kf, kmin=kf, kmax=0.1)     # Here, you may change the value of kmax
    Pk = r.power
    Pk_corr = Pk['power'].real
    return Pk['k'], Pk_corr

selection = {}  # Dictionary to store each bin

for i in range(len(bin_limits)):
    selection[f"selec_{i}"] = df.index[df['bin'] == i].to_numpy()
    N[i] = len(selection[f"selec_{i}"])

    bin_delta = df.iloc[selection[f"selec_{i}"]][['x', 'y', 'z', 'delta']].to_numpy()
    np.savetxt(f"/folder/to/store/bins/bin_delta_{i}.txt", bin_delta, header="x y z delta")

    k_halos, P_k = phh_compute(selection[f"selec_{i}"], x, y, z, Ngrid, BoxSize)
    route_pk_file = f"/folder/to/store/each/pk_halos_{i}.txt"
    np.savetxt(route_pk_file, np.column_stack((k_halos, P_k)), delimiter=" ", header="k P")

    # We reinitialize the P_k and k_halos array, to avoid any error
    P_k = []
    k_halos = []

np.savetxt("/path/to/store/N.txt", N)

In [ ]:
'''
Computation of the power spectrum of the matter, P_mm. For this, we need the density field.
'''

f_particles="path/to/density/field"
particles = BigFileMesh(f_particles,dataset='Field')
results = FFTPower(particles,mode='1d',dk=kf,kmin=kf, kmax=0.1)

Pk = results.power
Pk_corr = Pk['power'].real

np.savetxt('/path/to/store/pk_matter.txt',np.column_stack((Pk['k'], Pk_corr)) )

In [ ]:
'''
Computation of the cross power spectra. For this, we need the halo bins and the particles. 
Here we reload the bins from the file generated before, to be able to run each cell independently, 
but you may change the code to compute everything in one go.
'''

for i in range(len(bin_limits)):
    bin_delta = np.genfromtxt(f"/folder/to/store/bins/bin_delta_{i}.txt", 
                            delimiter=" ", names=True)
    x = bin_delta['x']
    y = bin_delta['y']
    z = bin_delta['z']
    pos = np.array([x,y,z]).T
    catalog = ArrayCatalog({'Position': pos})
    mesh = catalog.to_mesh(resampler='cic', compensated=True, Nmesh=Ngrid, BoxSize=BoxSize)
    r_cross = FFTPower(mesh, second=particles, mode='1d', dk=kf, kmin=kf, kmax=0.1)
    Pk_cross = r_cross.power
    Pk_cross_corr = Pk_cross['power'].real
    

    data = np.column_stack((Pk_cross['k'], Pk_cross_corr))

    route_pcross_file = f"/folder/to/store/each/pk_halos_{i}.txt"
    np.savetxt(route_pcross_file,
           data, header='k P')